# Feature Extraction 08: Sentence-Embedding Text Signal

Aggregates each message's pretrained sentence-embedding to stock-day level as new
x-variables, to test against next-day returns downstream. No model is trained and no
return label is used here -- each message's embedding is mean-pooled directly, the same
way `net_sentiment` (features_01) averages a discrete sentiment tag, except the raw
384-dim embedding vector stands in for the tag.

**Relationship to other notebooks**
- `features_06_full_text_exploration.ipynb` validated the `messages/` + `msg_info/` join
  and prototyped cashtag/mention extraction. This notebook reuses that same join logic
  (Sections 2-3 below) but works with the full message body instead of hand-extracted
  substrings.
- `features_07_user_influence_accuracy.ipynb` scores *users* by track record with a
  walk-forward, no-look-ahead design. This notebook does **not** need that discipline:
  because there's no model being fit on realized returns, there's no look-ahead risk to
  guard against -- each stock-day's aggregate depends only on that day's own messages.

**Method**
1. Encode each message's cleaned body with a pretrained sentence-transformer
   (`all-MiniLM-L6-v2`, 384-dim, L2-normalized).
2. Mean-pool the embedding vectors of every message posted on the same (`symbol`, `date`)
   into a single 384-dim vector for that stock-day, alongside the message count.
3. Save the resulting `embed_000_mean` ... `embed_383_mean` + `embed_n` columns as
   x-variables -- no training, no target label, no walk-forward.

**Scale warning -- read before running Section 8:** encoding is CPU-only in this
environment (no GPU detected: `torch` reports `cpu` build). Section 7 benchmarks
throughput on real sample data and extrapolates a wall-clock estimate for the full
corpus -- expect this to be on the order of days, not hours, of continuous compute.
Sections 3 and 8 are both checkpointed (per source file / per year) so the notebook can
be safely interrupted and resumed across multiple sessions.

## 1. Setup and Configuration

In [ ]:
import os
import time
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from sentence_transformers import SentenceTransformer

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
CODE_DIR      = Path(r"C:\Users\willi\.vscode\Github\ml-from-crowd")
DATA_DIR      = Path(r"E:\Research_data\Stocktwits\dataset\v1\data\csv")
MESSAGES_DIR  = DATA_DIR / "messages"                  # message_id, message_body (205 files, NOT year-chunked), message_id
#will comment: message_id is used to key off of other previously computed data, do not recompute the date!!!
RETURNS_DIR   = DATA_DIR / "merged_with_crsp_mlcrowd"  # message-level, per-year, has message_id + symbol/date
OUTPUT_FOLDER = DATA_DIR / "features_mlcrowd"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE   = OUTPUT_FOLDER / "features_08_text_embedding_signal.pkl"

# Intermediate/checkpoint storage -- large, kept separate from the small pickle outputs
WORK_DIR         = DATA_DIR / "features_08_workdir"
JOINED_DIR       = WORK_DIR / "joined_by_year"       # per-year csv: message_id, message_body, metadata
YEAR_FEATURE_DIR = WORK_DIR / "features_by_year"      # per-year aggregated stock-day features (resumable Section 8)
for d in (JOINED_DIR, YEAR_FEATURE_DIR):
    d.mkdir(parents=True, exist_ok=True)
JOIN_PASS_LOG = WORK_DIR / "join_pass_processed_files.txt"

# =============================================================================
# PARAMETERS
# =============================================================================
EMBED_MODEL_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM         = 384
EMBED_COLS        = [f"embed_{i:03d}" for i in range(EMBED_DIM)]
ENCODE_BATCH_SIZE = 256            # forward-pass batch size inside model.encode()
ENCODE_CHUNK_ROWS = 200_000        # max rows embedded into memory at once per year (see Section 5)

# Set to a list of years (e.g. [2010]) to smoke-test Section 8 end-to-end on just those
# years before committing to the full multi-day run; None processes every year found in
# JOINED_DIR. Has no effect on Sections 1-7, only on which years Section 8 loops over.
TEST_YEARS_ONLY = [2010]

print(f"Messages dir  : {MESSAGES_DIR}")
print(f"Returns dir   : {RETURNS_DIR}")
print(f"Output file   : {OUTPUT_FILE}")
print(f"Work dir      : {WORK_DIR}")
print(f"Embed model   : {EMBED_MODEL_NAME} ({EMBED_DIM}-dim)")
if TEST_YEARS_ONLY is not None:
    print(f"TEST_YEARS_ONLY: {TEST_YEARS_ONLY} (Section 8 will only process these years)")

import torch
print(f"torch device  : {'cuda' if torch.cuda.is_available() else 'cpu'} (build: {torch.__version__})")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'E:\\'

## 2. Build the Message Universe

`merged_with_crsp_mlcrowd/` already restricts to Bullish/Bearish-labeled, CRSP-matched
messages and is chunked by year, but it has no message text. Load just the join key and
the (`symbol`, `date`) it maps to across all years, so we know exactly which
`message_id`s are worth encoding and which year bucket each belongs to.

In [ ]:
RETURN_FILES = sorted(RETURNS_DIR.glob("stocktwits_crsp_*.csv"))fg
print(f"Found {len(RETURN_FILES)} year files in {RETURNS_DIR.name}")

NEEDED_COLS = ["message_id", "date", "symbol"]

universe_frames = []
for f in tqdm(RETURN_FILES, desc="Loading message universe"):
    year = int(f.stem.split("_")[-1])
    df_y = pd.read_csv(f, usecols=NEEDED_COLS)
    df_y["year"] = year
    universe_frames.append(df_y)

universe = pd.concat(universe_frames, ignore_index=True)
universe["date"] = pd.to_datetime(universe["date"])
universe["message_id"] = pd.to_numeric(universe["message_id"], errors="coerce")
universe = universe.dropna(subset=["message_id"])
universe["message_id"] = universe["message_id"].astype("int64")
universe = universe.drop_duplicates(subset="message_id")

print(f"Universe size: {len(universe):,} messages across {universe['year'].nunique()} years")
print(universe.groupby("year").size())

## 3. Pass 1 -- Join Message Text into the Universe (checkpointed)

`messages/` is NOT year-chunked (per `features_06`'s exploration -- 205 files whose
`message_id` ranges span many calendar years). So this streams every `messages/msg_*.csv`
file once, inner-joins each chunk's `message_id` against `universe`, and appends matches to
a per-year CSV under `JOINED_DIR`. Finished source files are recorded in `JOIN_PASS_LOG`,
so re-running this cell after a full completion is a no-op, and interrupting mid-run only
wastes at most one file's work.

A small fraction of message bodies contain stray/unterminated quote characters that make
pandas' default C parser raise a hard `ParserError` ("buffer overflow") partway through a
file rather than just corrupting one row (as `features_06` saw on a non-chunked read).
`engine="python"` with `on_bad_lines="skip"` tolerates these and drops the offending rows
instead of crashing the whole join pass.

In [ ]:
def _load_processed_files():
    if JOIN_PASS_LOG.exists():
        return set(JOIN_PASS_LOG.read_text().splitlines())
    return set()

def _mark_processed(fname):
    with open(JOIN_PASS_LOG, "a") as fh:
        fh.write(fname + "\n")

universe_indexed = universe.set_index("message_id")
message_files = sorted(MESSAGES_DIR.glob("msg_*.csv"))
processed = _load_processed_files()
todo_files = [f for f in message_files if f.name not in processed]

print(f"messages/ files: {len(message_files)} total, {len(processed)} already joined, "
      f"{len(todo_files)} remaining")

CHUNK_SIZE = 200_000
_open_writers = {}  # year -> (file handle, still-need-header bool)

def _get_writer(year):
    if year not in _open_writers:
        path = JOINED_DIR / f"joined_{year}.csv"
        needs_header = not path.exists()
        _open_writers[year] = [open(path, "a", newline="", encoding="utf-8"), needs_header]
    return _open_writers[year]

for f in tqdm(todo_files, desc="Joining messages/ files"):
    # engine="python" + on_bad_lines="skip": a small fraction of message bodies contain
    # stray/unterminated quote characters that make the (much faster) C engine raise a
    # hard "buffer overflow" ParserError partway through the file (observed on msg_000.csv
    # around row ~600k). The python engine tolerates these and just drops the bad rows.
    reader = pd.read_csv(
        f, usecols=["message_id", "message_body"], chunksize=CHUNK_SIZE,
        engine="python", on_bad_lines="skip",
    )
    for chunk in reader:
        chunk["message_id"] = pd.to_numeric(chunk["message_id"], errors="coerce")
        chunk = chunk.dropna(subset=["message_id"])
        chunk["message_id"] = chunk["message_id"].astype("int64")

        joined = chunk.join(universe_indexed, on="message_id", how="inner")
        if joined.empty:
            continue

        for year, grp in joined.groupby("year"):
            writer = _get_writer(int(year))
            grp.drop(columns="year").to_csv(writer[0], header=writer[1], index=False)
            writer[1] = False

    _mark_processed(f.name)

for fh, _ in _open_writers.values():
    fh.close()

print("Join pass complete (or already complete from a prior run).")
print("Joined year files:", sorted(p.name for p in JOINED_DIR.glob("joined_*.csv")))

## 4. Core Functions: Text Cleaning, Embedding, Aggregation

In [ ]:
import html
import re

WHITESPACE_RE = re.compile(r"\s+")

def clean_text(body):
    # Unescape HTML entities and collapse whitespace; keep cashtags/mentions as
    # context for the embedding model rather than stripping them.
    return WHITESPACE_RE.sub(" ", html.unescape(str(body))).strip()

def load_year(year):
    path = JOINED_DIR / f"joined_{year}.csv"
    df = pd.read_csv(path)
    df["message_id"] = df["message_id"].astype("int64")
    df["date"] = pd.to_datetime(df["date"])
    df["message_body"] = df["message_body"].apply(clean_text)
    return df

def embed_texts(model, texts, batch_size=ENCODE_BATCH_SIZE):
    return model.encode(
        list(texts), batch_size=batch_size, show_progress_bar=False,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")

def aggregate_chunk(sub, embeddings):
    # Partial per-(symbol, date) sums for one chunk of one year. Combined across
    # chunks and divided by count in finalize_aggregation() to get the mean-pooled
    # embedding -- summing first (rather than averaging per chunk) keeps the result
    # exact regardless of how a year happens to be split into chunks.
    chunk_df = pd.DataFrame(embeddings, columns=EMBED_COLS)
    chunk_df["symbol"] = sub["symbol"].to_numpy()
    chunk_df["date"] = sub["date"].to_numpy()
    chunk_df["embed_n"] = 1
    return chunk_df.groupby(["symbol", "date"], as_index=False).sum()

#Will comment: this is the primary function for first pass sum sentence transformer data by symbol date pairings 

def finalize_aggregation(partial_aggs):
    combined = pd.concat(partial_aggs, ignore_index=True)
    totals = combined.groupby(["symbol", "date"], as_index=False).sum()
    n = totals["embed_n"]
    for col in EMBED_COLS:
        totals[col] = totals[col] / n
    return totals.rename(columns={c: f"{c}_mean" for c in EMBED_COLS})

#Will comment: this sums across processing chunks and then divides by count to get mean embedding values.

## 5. Aggregation Driver

For year `Y`, encode messages in row-chunks of `ENCODE_CHUNK_ROWS` (keeping peak memory
bounded regardless of how large a single year is), aggregating each chunk to
per-(`symbol`, `date`) sums immediately and discarding the embeddings before moving to
the next chunk. The per-chunk partial sums are combined and divided by count once the
whole year has been encoded.

Unlike a supervised walk-forward model, this aggregation has no look-ahead risk to guard
against: a stock-day's mean embedding depends only on that day's own messages, not on any
other year's data, so years can be processed in any order (or even in parallel) --
processing them in order here is just a convenient way to checkpoint progress
(Section 8).

`SentenceTransformer.encode()` batches the forward pass internally (`ENCODE_BATCH_SIZE`),
but it still accumulates *every* resulting embedding into one array before returning --
for a year with 25M+ messages that's a 36+ GiB float32 array, which is exactly what
crashed with `MemoryError` on the largest year (2021, the meme-stock volume spike) in
practice. Chunking at the outer level with `ENCODE_CHUNK_ROWS` caps peak memory.

In [ ]:
def process_year(year, model, chunk_rows=ENCODE_CHUNK_ROWS):
    df = load_year(year)
    n = len(df)
    n_chunks = math.ceil(n / chunk_rows)

    partial_aggs = []
    encode_secs = 0.0

    for chunk_i, start in enumerate(range(0, n, chunk_rows)):
        end = min(start + chunk_rows, n)
        sub = df.iloc[start:end]

        t0 = time.time()
        embeddings = embed_texts(model, sub["message_body"].tolist())
        encode_secs += time.time() - t0

        partial_aggs.append(aggregate_chunk(sub, embeddings))
        #Will comment: this is where the first sum is called for the chunk
        del embeddings  # free before encoding the next chunk

        if n_chunks > 1:
            print(f"    {year}: chunk {chunk_i + 1}/{n_chunks} ({end:,}/{n:,} rows) "
                  f"cumulative encode time {encode_secs:,.0f}s")

    agg = finalize_aggregation(partial_aggs)
    #Will comment: this is where the final average is calculated
    agg["year"] = year

    stats = {
        "year": year, "n_messages": n, "n_stock_days": len(agg), "n_chunks": n_chunks,
        "encode_secs": encode_secs, "msgs_per_sec": n / max(encode_secs, 1e-9),
    }
    return agg, stats

## 6. Validate on a Single Year (2010 -- the smallest year)

Runs the real pipeline end-to-end (embedding model, chunked encode, per-(symbol, date)
aggregation) on the smallest available year before committing to the full run.

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")

years_available = sorted(int(p.stem.split("_")[-1]) for p in JOINED_DIR.glob("joined_*.csv"))
print(f"Years available after the join pass: {years_available}")

first_year = years_available[0]
agg_first, stats_first = process_year(first_year, embed_model)

print(f"\nYear {first_year}:")
print(stats_first)
display(agg_first.head(10))
assert agg_first.shape[1] == EMBED_DIM + 3, "Expected symbol, date, embed_n + EMBED_DIM mean columns"
assert agg_first["embed_n"].min() >= 1, "Every stock-day should be backed by at least one message"
print("\nValidation passed: chunked encode + per-(symbol, date) mean-pooling works.")

## 7. Runtime Benchmark & Full-Corpus Estimate

**Read this before running Section 8.** Encoding is CPU-only here. This benchmarks
observed throughput from Section 6 and extrapolates a wall-clock estimate for the full
`universe` built in Section 2, so the actual cost is known before committing to it.

In [ ]:
total_messages = len(universe)
observed_rate = stats_first["msgs_per_sec"]  # from the real validation run, not a toy benchmark

est_seconds = total_messages / observed_rate
est_hours = est_seconds / 3600

print(f"Observed encoding rate ({first_year}, CPU): {observed_rate:,.1f} messages/sec")
print(f"Total messages in universe: {total_messages:,}")
print(f"Estimated full-corpus encoding time: {est_hours:,.1f} hours (~{est_hours/24:,.1f} days)")
print()
print("This is a lower bound: it excludes I/O for the join pass (Section 3, ~52 GB of raw")
print("text, already a one-time cost) and assumes throughput stays constant across years")
print("(later years have far more messages per file, so cache/memory pressure may differ).")
print()
print("Section 8 is checkpointed per year via YEAR_FEATURE_DIR: it is safe to run it for a")
print("while, stop, and resume later across multiple sessions.")

## 8. Process All Years (Checkpointed)

Resumable at year granularity: any year whose feature file already exists under
`YEAR_FEATURE_DIR` is skipped. Because aggregation has no cross-year state (no model to
keep in sync), resuming is simply "skip what's already on disk" -- there's no risk of a
year silently missing an update the way an interrupted walk-forward fit would be.

Per-year features are cached as pickle rather than parquet: it needs no extra dependency
(pyarrow/fastparquet), matches every other feature notebook in this project, and sidesteps
a pandas/pyarrow Arrow-extension-type registry incompatibility (`ArrowKeyError: No type
extension with name arrow.py_extension_type found`) observed with newer pyarrow releases
against pandas 2.2.3 in this environment.

In [ ]:
def _year_feature_path(year):
    return YEAR_FEATURE_DIR / f"features_{year}.pkl"

years_available = sorted(int(p.stem.split("_")[-1]) for p in JOINED_DIR.glob("joined_*.csv"))
if TEST_YEARS_ONLY is not None:
    years_available = [y for y in years_available if y in TEST_YEARS_ONLY]
    print(f"TEST_YEARS_ONLY active: restricting this run to {years_available}")

done_years = sorted(int(p.stem.split("_")[-1]) for p in YEAR_FEATURE_DIR.glob("features_*.pkl"))
remaining_years = [y for y in years_available if y not in done_years]

print(f"Years already done: {done_years}")
print(f"Years remaining to process: {remaining_years}")

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
run_stats = []

for year in tqdm(remaining_years, desc="Aggregating years"):
    agg, stats = process_year(year, embed_model)
    agg.to_pickle(_year_feature_path(year))
    run_stats.append(stats)
    print(f"  {year}: {stats['n_messages']:,} msgs -> {stats['n_stock_days']:,} stock-days "
          f"({stats['msgs_per_sec']:.1f} msg/s)")

print("\nAll years processed (or resumed to completion).")
if run_stats:
    display(pd.DataFrame(run_stats))

## 9. Combine All Years & Final Inspection

In [ ]:
year_files = sorted(YEAR_FEATURE_DIR.glob("features_*.pkl"))
features_all = pd.concat([pd.read_pickle(f) for f in year_files], ignore_index=True)
features_all = features_all.drop(columns="year").sort_values(["date", "symbol"]).reset_index(drop=True)

print(f"Shape: {features_all.shape}")
print(f"Columns: {list(features_all.columns)}")
print(f"Date range: {features_all['date'].min().date()} to {features_all['date'].max().date()}")
print(f"Unique symbols: {features_all['symbol'].nunique()}")
print(f"\nNull counts:")
print(features_all.isnull().sum())
print(f"\nSummary statistics:")
display(features_all.describe().round(4))

if TEST_YEARS_ONLY is not None:
    print(f"\nWARNING: TEST_YEARS_ONLY = {TEST_YEARS_ONLY} -- this is a partial smoke-test "
          f"combine, not the full corpus. Do NOT run Section 10 to overwrite {OUTPUT_FILE.name} "
          f"with this. Set TEST_YEARS_ONLY = None and re-run Section 8 for the real run.")

## 10. Save to Pickle

In [ ]:
assert TEST_YEARS_ONLY is None, (
    f"TEST_YEARS_ONLY = {TEST_YEARS_ONLY} -- refusing to save a partial smoke-test combine "
    f"over {OUTPUT_FILE.name}. Set TEST_YEARS_ONLY = None and re-run Section 8 for the real run."
)

print(f"Saving to: {OUTPUT_FILE}")
features_all.to_pickle(OUTPUT_FILE)

verify = pd.read_pickle(OUTPUT_FILE)
assert verify.shape == features_all.shape
file_mb = OUTPUT_FILE.stat().st_size / 1024**2
print(f"Saved and verified. File size: {file_mb:.1f} MB")

## Summary

**Rationale**

Every other feature in this project treats a message's self-reported `sentiment` tag
(Bullish/Bearish) as the unit of signal. That discards the actual text -- two "Bullish"
messages can carry very different information (a one-word "$AAPL bullish" vs. a detailed
thesis). This notebook lets the raw text speak through a pretrained sentence embedding,
aggregated to stock-day level, as a model-free alternative (or complement) to the
hand-labeled sentiment tag.

**Methodology**

1. Each message is embedded with a small pretrained sentence-transformer
   (`all-MiniLM-L6-v2`, 384-dim, L2-normalized output).
2. Message-level embeddings are mean-pooled to (`symbol`, `date`): `embed_000_mean`
   through `embed_383_mean` (the crowd's average text-embedding that day) plus `embed_n`
   (message count backing the aggregate).
3. No model is trained and no return label is used -- this is pure aggregation, so there
   is no look-ahead risk and years can be processed independently/in any order.

**Known limitations**

- Mean-pooling 384 raw embedding dimensions is not directly interpretable the way a
  scalar sentiment score is; downstream use likely means treating these as inputs to a
  model (e.g. PCA or a supervised learner in `03a`-`03e`) rather than reading them
  directly.
- Encoding is CPU-only in this environment; see Section 7's runtime estimate before
  running Section 8 on the full corpus.

**Next steps**

- Merge `features_08_text_embedding_signal.pkl` into `merged_master.pkl` alongside the
  other feature files (`02 - prepare training dataset/merge_all_feature_files.ipynb`).
- Decide how downstream notebooks consume 384 embedding columns (e.g. dimensionality
  reduction) before comparing against `net_sentiment` (features_01) and
  `skill_wtd_net_sent_*` (features_07).
- Per `features_06`'s roadmap, this is one of several open-ended full-text directions
  (topic modeling, sentiment-lexicon validation); those remain unexplored.